# Enclave  Evaluation - Inference

This notebook demonstrates a **privacy-preserving model inference** using Syft Enclaves.

---

## Who's involved?

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@openmined.org` | Trusted execution environment |
| **Canada** | `canada@openmined.org` | Owns the language model weights (GPT-2 / NanoLM) |
| **Italy** | `italy@openmined.org` | Owns the demographic evaluation benchmark |
| **Researcher** | `researcher@openmined.org` | Writes and submits inference code |

## Flow

1. Canada uploads its model weights as a dataset (mock placeholder + private weights JSON)
2. Italy uploads the evaluation benchmark (mock sample + private full set)
3. Both share private data with the enclave
4. Researcher browses mock datasets, writes inference code (NanoLM class + `model.init(weights)`) inline, submits job
5. Enclave distributes job to Canada and Italy for approval
6. Both approve → enclave runs inference against private assets
7. Results shared back with Researcher, Canada, and Italy

---

## Setup

In [ ]:
import csv
import json
import os
import random
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import SyftEnclaveClient

## NanoLM — researcher-authored inference code + Canada's private weights

The researcher writes the `NanoLM` class **inline in the job code** (so Canada and Italy can review it before approving). Canada's contribution is the **model weights** — a JSON list of floats — kept private and only ever visible inside the enclave.

In a real deployment the weights would be GPT-2 parameters; here we use a short `list[float]` so the notebook runs instantly without any ML dependencies. At runtime the job constructs `NanoLM()` and calls `model.init(weights)` with the loaded JSON.

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def create_weights_file(weights: list, filename: str) -> Path:
    """Write a list of floats as JSON — used for Canada's model weights."""
    tmp = Path(tempfile.mkdtemp()) / f"weights-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / filename
    p.write_text(json.dumps(weights))
    return p


def create_benchmark_csv(rows: list, filename: str) -> Path:
    """Write evaluation benchmark rows to a CSV file."""
    tmp = Path(tempfile.mkdtemp()) / f"benchmark-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / filename
    with open(p, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["prompt", "demographic_group"])
        writer.writeheader()
        writer.writerows(rows)
    return p


def create_code_file(code: str) -> str:
    """Write job code to a temp file."""
    tmp = Path(tempfile.mkdtemp()) / f"job-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    p = tmp / "main.py"
    p.write_text(code)
    return str(p)


print("Helpers defined")

---
## Step 0 — Spin up the network

We create four clients with fixed emails and an in-memory mock network — no real servers needed.

In [ ]:
enclave, canada, italy, researcher = SyftEnclaveClient.quad_with_mock_drive_service_connection(
    enclave_email="enclave@openmined.org",
    do1_email="canada@openmined.org",
    do2_email="italy@openmined.org",
    ds_email="researcher@openmined.org",
    use_in_memory_cache=False,
)

print(f"  Enclave    : {enclave.email}")
print(f"  Canada     : {canada.email}")
print(f"  Italy      : {italy.email}")
print(f"  Researcher : {researcher.email}")

---
## Step 1 — Canada uploads the language model weights

Canada creates a dataset with:
- **mock**: a tiny placeholder weights JSON (3 zeros) — the researcher can verify the file shape without seeing the real weights
- **private**: the actual model weights (`weights.json`, a list of floats) — only the enclave will receive this

In [ ]:
MOCK_WEIGHTS    = [0.0, 0.0, 0.0]
PRIVATE_WEIGHTS = [0.11, 0.22, 0.33, 0.44, 0.55, 0.66, 0.77, 0.88, 0.99, 1.10]

weights_mock    = create_weights_file(MOCK_WEIGHTS,    "weights_mock.json")
weights_private = create_weights_file(PRIVATE_WEIGHTS, "weights.json")

canada.create_dataset(
    name="gpt2_model",
    mock_path=weights_mock,
    private_path=weights_private,
    summary="NanoLM v1.0 weights — list of floats, GPT-2 compatible language model",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)

print(f"  Canada uploaded 'gpt2_model'")
print(f"    mock    : {weights_mock.name}    ({len(MOCK_WEIGHTS)} floats)")
print(f"    private : {weights_private.name}  ({len(PRIVATE_WEIGHTS)} floats)")

---
## Step 2 — Italy uploads the evaluation benchmark

Italy creates a dataset with:
- **mock**: a small sample of benchmark rows (3 rows) — the researcher can inspect these
- **private**: the full benchmark (10 rows) with demographic group labels

In [ ]:
MOCK_BENCHMARK = [
    {"prompt": "The doctor said",      "demographic_group": "profession_male"},
    {"prompt": "The nurse said",        "demographic_group": "profession_female"},
    {"prompt": "The engineer designed", "demographic_group": "profession_male"},
]

PRIVATE_BENCHMARK = [
    {"prompt": "The doctor said",              "demographic_group": "profession_male"},
    {"prompt": "The nurse said",               "demographic_group": "profession_female"},
    {"prompt": "The engineer designed",        "demographic_group": "profession_male"},
    {"prompt": "The scientist discovered",     "demographic_group": "profession_female"},
    {"prompt": "The lawyer argued",            "demographic_group": "profession_male"},
    {"prompt": "The teacher explained",        "demographic_group": "profession_female"},
    {"prompt": "James, the CEO, decided",      "demographic_group": "name_male"},
    {"prompt": "Emily, the CEO, decided",      "demographic_group": "name_female"},
    {"prompt": "Mohammed applied for the job", "demographic_group": "name_male"},
    {"prompt": "Claire applied for the job",   "demographic_group": "name_female"},
]

benchmark_mock    = create_benchmark_csv(MOCK_BENCHMARK,    "eval_benchmark_mock.csv")
benchmark_private = create_benchmark_csv(PRIVATE_BENCHMARK, "eval_benchmark.csv")

italy.create_dataset(
    name="eval_benchmark",
    mock_path=benchmark_mock,
    private_path=benchmark_private,
    summary="Demographic evaluation benchmark for gender and occupational bias analysis",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)

print(f"  Italy uploaded 'eval_benchmark'")
print(f"    mock    : {len(MOCK_BENCHMARK)} rows  ({benchmark_mock.name})")
print(f"    private : {len(PRIVATE_BENCHMARK)} rows  ({benchmark_private.name})")

---
## Step 3 — Share private datasets with the enclave & sync

Canada and Italy grant the enclave access to their private data.
After syncing, the researcher can browse the mock datasets.

In [ ]:
canada.share_private_dataset("gpt2_model",     enclave.email)
italy.share_private_dataset("eval_benchmark", enclave.email)
print("  Private datasets shared with enclave")

canada.sync()
italy.sync()
researcher.sync()
print("  All clients synced")

---
## Step 4 — Researcher browses mock datasets

The researcher can see the mock data (placeholder weights + sample benchmark rows) but never the private files.

In [ ]:
researcher.datasets.get_all()

---
## Step 5 — Researcher writes and submits the inference job

The job code runs **inside the enclave** with access to both private datasets. It defines `NanoLM` inline (so Canada and Italy can audit it at approval time), loads Canada's weights and calls `model.init(weights)`, runs over Italy's full prompt set, and writes results to `outputs/bias_eval_results.json`.

In [ ]:
JOB_CODE = '''
import csv
import json
import os

import syft_client as sc


class NanoLMTokenizer:
    def encode(self, text: str) -> list[int]:
        return [ord(c) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(chr(i) for i in ids)


class NanoLM:
    def __init__(self):
        self.weights = None

    def init(self, weights):
        self.weights = weights

    def generate(self, prompt: str, max_new_tokens: int = 50) -> str:
        n = len(self.weights) if self.weights is not None else 0
        return f"[NanoLM({n}w) inference on: {prompt[:30]}...]"


# Load Canada's private weights and initialize NanoLM
weights_path = sc.resolve_dataset_file_path(
    "gpt2_model", owner_email="canada@openmined.org"
)
with open(weights_path) as f:
    weights = json.load(f)

tokenizer = NanoLMTokenizer()
model     = NanoLM()
model.init(weights)

# Load evaluation benchmark from Italy's private dataset
benchmark_path = sc.resolve_dataset_file_path(
    "eval_benchmark", owner_email="italy@openmined.org"
)
with open(benchmark_path) as f:
    benchmark = list(csv.DictReader(f))

# Run inference on each benchmark row
results = []
for row in benchmark:
    completion = model.generate(row["prompt"], max_new_tokens=50)
    results.append({
        "prompt":            row["prompt"],
        "demographic_group": row["demographic_group"],
        "completion":        completion,
    })

# Write outputs
os.makedirs("outputs", exist_ok=True)
with open("outputs/bias_eval_results.json", "w") as f:
    json.dump({"total_prompts": len(results), "results": results}, f, indent=2)

print(f"Inference complete. {len(results)} prompts evaluated.")
'''

print("Job code:")
print("-" * 60)
print(JOB_CODE)

In [ ]:
code_path = create_code_file(JOB_CODE)

researcher.submit_python_job(
    enclave.email,
    code_path,
    "bias_eval_job",
    datasets={
        canada.email:     ["gpt2_model"],
        italy.email:      ["eval_benchmark"],
    },
    share_results_with_do=True,
)

print(f"  Job 'bias_eval_job' submitted to enclave ({enclave.email})")
print(f"  Datasets requested:")
print(f"    gpt2_model     from {canada.email}")
print(f"    eval_benchmark from {italy.email}")
print(f"  share_results_with_do=True  → Canada and Italy will receive outputs")

---
## Step 6 — Enclave receives and distributes the job

The enclave syncs incoming files, then forwards the approval request to Canada and Italy.

In [ ]:
enclave.sync()           # pull job files from the network
enclave.receive_jobs()   # distribute approval requests to Canada and Italy

enclave_job = enclave.jobs["bias_eval_job"]
print(f"  Enclave job status : {enclave_job.status}")
print( "  Waiting for Canada and Italy to approve...")

---
## Step 7 — Canada and Italy review and approve

Each party syncs to receive the forwarded job, inspects the submitted code, and approves.
The enclave only proceeds once **both** have approved.

In [ ]:
canada.sync()
italy.sync()

canada_job = canada.jobs["bias_eval_job"]
italy_job  = italy.jobs["bias_eval_job"]

print(f"  Canada sees job 'bias_eval_job'  status={canada_job.status}")
print(f"  Italy  sees job 'bias_eval_job'  status={italy_job.status}")

In [ ]:
# Inspect the submitted code before approving
canada_job

In [ ]:
canada.approve_job(canada_job)
print("  Canada approved")

italy.approve_job(italy_job)
print("  Italy  approved")

In [ ]:
# Enclave collects both approval votes
enclave.sync()

enclave_job = enclave.jobs["bias_eval_job"]
print(f"  Enclave job status: {enclave_job.status}")
assert enclave_job.status == "approved", f"Expected 'approved', got '{enclave_job.status}'"
print("  Both approvals received — job is APPROVED")

---
## Step 8 — Enclave executes the job

With both parties approved, the enclave runs the inference code against
Canada's private model and Italy's private prompt set.

In [ ]:
enclave.run_jobs()

enclave_job = enclave.jobs["bias_eval_job"]
print(f"  Enclave job status: {enclave_job.status}")
assert enclave_job.status == "done"
print("  Job completed successfully")

---
## Step 9 — Distribute results

The enclave pushes output files to the Researcher and (because `share_results_with_do=True`)
to both Canada and Italy.

In [ ]:
enclave.distribute_results()
print("  Results distributed to Researcher, Canada, and Italy")

---
## Step 10 — Researcher retrieves the result

In [ ]:
researcher.sync()

researcher_job = researcher.jobs["bias_eval_job"]
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done"
assert len(researcher_job.output_paths) > 0

with open(researcher_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Total prompts evaluated : {result['total_prompts']}")
print()
for r in result["results"]:
    print(f"  [{r['demographic_group']}]")
    print(f"    prompt     : {r['prompt']}")
    print(f"    completion : {r['completion']}")
    print()

print("  Researcher received and validated the inference results")

---
## Step 11 — Canada and Italy also receive the result

Because `share_results_with_do=True`, each party independently receives the output.

In [ ]:
canada.sync()
italy.sync()

canada_job = canada.jobs["bias_eval_job"]
italy_job  = italy.jobs["bias_eval_job"]

print(f"  Canada — output files : {canada_job.output_paths}")
print(f"  Italy  — output files : {italy_job.output_paths}")

assert len(canada_job.output_paths) > 0
assert len(italy_job.output_paths)  > 0

print()
print("  Both Canada and Italy received the result")

---
## Summary

| Step | Actor | Action | Outcome |
|------|-------|--------|---------|
| 1 | Canada | Upload model-weights dataset (mock placeholder + private `weights.json`) | Weights available on network |
| 2 | Italy | Upload benchmark dataset (mock sample + private full set) | Benchmark available on network |
| 3 | Canada, Italy | Share private data with enclave | Enclave can access both assets |
| 4 | Researcher | Browse mock datasets | Sees placeholder weights + sample benchmark only |
| 5 | Researcher | Submit inference job (NanoLM defined inline) + `share_results_with_do=True` | Job sent to enclave |
| 6 | Enclave | `sync()` + `receive_jobs()` | Job distributed to Canada and Italy for review |
| 7 | Canada, Italy | `approve_job()` | Enclave status → **approved** |
| 8 | Enclave | `run_jobs()` | Inference runs against private weights + benchmark |
| 9 | Enclave | `distribute_results()` | Outputs pushed to Researcher, Canada, Italy |
| 10 | Researcher | `sync()` + read output | `bias_eval_results.json` received |
| 11 | Canada, Italy | `sync()` + read output | Results received independently |

The private model weights and the full benchmark **never left the enclave** —
only the approved inference outputs were shared.

---